<a href="https://colab.research.google.com/github/Adellia03/Pembelajaran-Mesin/blob/main/JS03/JS03_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("/content/drive/MyDrive/Machine Learning/wbc.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [50]:
#1. Pisahkan antara variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.

df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

print("Variabel yang dapat digunakan:")
print(X.columns.tolist())

print("Variabel yang tidak dapat digunakan:")
print("id")

Variabel yang dapat digunakan:
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']
Variabel yang tidak dapat digunakan:
id


In [51]:
#2. Lakukan proses encoding pada kolom "diagnosis".

le = LabelEncoder()

df['diagnosis'] = le.fit_transform(df['diagnosis'])

print("Hasil encoding diagnosis:")
print(df['diagnosis'].value_counts())

print("Keterangan encoding:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

Hasil encoding diagnosis:
diagnosis
0    357
1    212
Name: count, dtype: int64
Keterangan encoding:
{'B': np.int64(0), 'M': np.int64(1)}


In [53]:
#3. Lakukan proses standardisasi pada semua kolom yang memiliki nilai numerik.
scaler = StandardScaler()

X_scaled = pd.DataFrame(scaler.fit_transform(X),columns=X.columns)

print("Data setelah standardisasi:")
display(X_scaled.head())

Data setelah standardisasi:


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1.097064,-2.073335,1.269934,0.984375,1.568466,3.283515,2.652874,2.532475,2.217515,2.255747,...,1.886690,-1.359293,2.303601,2.001237,1.307686,2.616665,2.109526,2.296076,2.750622,1.937015
1,1.829821,-0.353632,1.685955,1.908708,-0.826962,-0.487072,-0.023846,0.548144,0.001392,-0.868652,...,1.805927,-0.369203,1.535126,1.890489,-0.375612,-0.430444,-0.146749,1.087084,-0.243890,0.281190
2,1.579888,0.456187,1.566503,1.558884,0.942210,1.052926,1.363478,2.037231,0.939685,-0.398008,...,1.511870,-0.023974,1.347475,1.456285,0.527407,1.082932,0.854974,1.955000,1.152255,0.201391
3,-0.768909,0.253732,-0.592687,-0.764464,3.283553,3.402909,1.915897,1.451707,2.867383,4.910919,...,-0.281464,0.133984,-0.249939,-0.550021,3.394275,3.893397,1.989588,2.175786,6.046041,4.935010
4,1.750297,-1.151816,1.776573,1.826229,0.280372,0.539340,1.371011,1.428493,-0.009560,-0.562450,...,1.298575,-1.466770,1.338539,1.220724,0.220556,-0.313395,0.613179,0.729259,-0.868353,-0.397100


In [74]:
#4. Lakukan proses seleksi fitur. Anda dapat menggunakan SelectKBest.
hasil = []

for k in range(1, X_scaled.shape[1] + 1):
    selector = SelectKBest(
    score_func=f_classif, k=k)
    X_selected = selector.fit_transform(X_imputed, y)

    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42) #Membagi data training dan testing

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    hasil.append([k, accuracy])

hasil_df = pd.DataFrame(hasil, columns=['Jumlah Fitur', 'Accuracy'])

print("Hasil seleksi fitur:")
display(hasil_df)

Hasil seleksi fitur:


,Jumlah Fitur,Accuracy
0,1,0.929825
1,2,0.956140
2,3,0.956140
3,4,0.973684
4,5,0.973684
5,6,0.973684
6,7,0.973684
7,8,0.973684
8,9,0.973684
9,10,0.973684


In [75]:
#5. Lakukan proses pengujian dengan model Logistic Regression seperti pada praktikum 1.

index_terbaik = hasil_df['Accuracy'].idxmax()
k_terbaik = int(hasil_df.loc[index_terbaik, 'Jumlah Fitur'])
accuracy_terbaik = hasil_df.loc[index_terbaik, 'Accuracy']

print("Jumlah fitur terbaik:", k_terbaik)
print("Accuracy terbaik:", accuracy_terbaik)

selector = SelectKBest(score_func=f_classif, k=k_terbaik)
X_selected = selector.fit_transform(X_imputed, y)
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy Logistic Regression:", accuracy)

Jumlah fitur terbaik: 18
Accuracy terbaik: 0.9824561403508771
Accuracy Logistic Regression: 0.9824561403508771


In [56]:
#6. Anda dapat menggunakan model pipeline untuk mempermudah perkejaan Anda.

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=k_terbaik)),
    ('model', LogisticRegression(max_iter=1000))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)
y_pred_pipeline = pipeline.predict(X_test)
accuracy_pipeline = accuracy_score(y_test, y_pred_pipeline)

print("Accuracy menggunakan Pipeline:", accuracy_pipeline)

Accuracy menggunakan Pipeline: 0.9824561403508771


In [76]:
#7. Berdasarkan hasil analisa Anda, berapa jumlah fitur terbaik yang dapat digunakan? Apa saja fitur tersebut?
fitur_terbaik = X.columns[selector.get_support()]

print("Jumlah fitur terbaik:", k_terbaik)

print("Fitur yang terpilih:")
for i, fitur in enumerate(fitur_terbaik, 1): print(i, fitur)

print("Accuracy terbaik:", accuracy_terbaik)

Jumlah fitur terbaik: 18
Fitur yang terpilih:
1 radius_mean
2 perimeter_mean
3 area_mean
4 compactness_mean
5 concavity_mean
6 concave points_mean
7 radius_se
8 perimeter_se
9 area_se
10 radius_worst
11 texture_worst
12 perimeter_worst
13 area_worst
14 smoothness_worst
15 compactness_worst
16 concavity_worst
17 concave points_worst
18 symmetry_worst
Accuracy terbaik: 0.9824561403508771
